In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as hc
import scipy.spatial as sp

import matplotlib
import matplotlib.patches as patches
from matplotlib import pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm.notebook import tqdm

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
from kneebow.rotor import Rotor

In [ ]:
RAW_GENOMES = '../../data/raw/genomes'
MASH_GENOMES = '../../data/raw/mash_genomes'

In [ ]:
SCRUBBED_SUMMARY = '../../data/metadata/scrubbed_species_summary.csv'
SCRUBBED_METADATA = '../../data/metadata/scrubbed_species_metadata.csv'

In [ ]:
scrubbed_summary = pd.read_csv(SCRUBBED_SUMMARY, index_col=0, dtype='object')
scrubbed_metadata = pd.read_csv(SCRUBBED_METADATA, index_col=0, dtype='object')

# Ensure genome id is a str
scrubbed_summary['genome_id'] = scrubbed_summary['genome_id'].astype('str')
scrubbed_metadata['genome_id'] = scrubbed_metadata['genome_id'].astype('str')

# fix naming format issues
scrubbed_metadata['genome_name'] = scrubbed_metadata['genome_name'].apply(lambda x: x.replace('"[Enterobacter]', "Enterobacter"))
scrubbed_metadata['genome_name'] = scrubbed_metadata['genome_name'].apply(lambda x: x.replace('"', ""))

display(
    scrubbed_metadata.shape,
    scrubbed_metadata.head()
)

In [ ]:
# Make list of genome paths to run through fastANI
items = []
item_paths = []

for item in os.listdir('../../data/raw/mash_genomes/'):
    if item[0] == '.': continue
    curr_path = os.path.join('../../data/raw/mash_genomes/', item)
    items.append(item[:-4])
    item_paths.append(curr_path)
    # if os.path.isdir(curr_path):
    #     curr_fna = os.path.join(curr_path, f'{item}.fna')
    #     items.append(item)
    #     item_paths.append(curr_fna)


display(
    items[:5],
    item_paths[:5]
)

In [ ]:
scrubbed_metadata

In [ ]:
with open('../../data/tmp/fasnani_paths.txt', 'w') as f:
    for item, path in zip(items, item_paths):
        if item in scrubbed_metadata.genome_id.values:
            f.write(path + '\n')

## Run FastANI

Linux tmux session:

```
fastANI --ql ../../data/tmp/fasnani_paths.txt --rl ../../data/tmp/fasnani_paths.txt -o ../../data/processed/fastani_output/fastani_output.out -t 60
```

# ANI filtration and clustering

In [ ]:
names = [
    'genome1',
    'genome2',
    'ani',
    'hits',
    'total_fragments'
]

df_fastani = pd.read_csv('../../data/processed/fastani_output/fastani_output.out', sep='\t', names=names)
df_fastani['genome1'] = df_fastani['genome1'].apply(lambda x: x.split('/')[-1].split('.fna')[0])
df_fastani['genome2'] = df_fastani['genome2'].apply(lambda x: x.split('/')[-1].split('.fna')[0])

df_fastani

In [ ]:
df_ani_square = df_fastani.pivot(index='genome1', columns='genome2', values='ani') / 100

# make the matrix symmetrical
df_ani_square = df_ani_square.loc[df_ani_square.index, df_ani_square.index]
arr = df_ani_square.values
sym_arr = (arr + arr.T) / 2
df_ani_square = pd.DataFrame(sym_arr, index=df_ani_square.index, columns=df_ani_square.columns)

display(
    df_ani_square.shape,
    df_ani_square.head()
)

In [ ]:
# # Once finished it will IMMEDIATELY save all 3 matrices
# # so you don't have to re-compute this over and over again

# df_ani_corr = df_ani_square.corr()
# df_ani_corr_dist = 1 - df_ani_corr
# df_ani_corr_dist

# # Save matrix so the next time, only the following cell needs to be run
# # This cell should be commented out after being run once
# df_ani_corr_dist.to_csv('../../data/processed/fastani_output/df_ani_corr_dist.csv')
# df_ani_square.to_csv('../../data/processed/fastani_output/df_ani_square.csv')
# df_ani_corr.to_csv('../../data/processed/fastani_output/df_ani_corr.csv')

# display(
#     df_ani_corr_dist.shape,
#     df_ani_corr_dist.head()
# )

In [ ]:
df_ani_corr_dist = pd.read_csv('../../data/processed/fastani_output/df_ani_corr_dist.csv', dtype='object').set_index('genome1').astype(float)
df_ani_square = pd.read_csv('../../data/processed/fastani_output/df_ani_square.csv', dtype='object').set_index('genome1').astype(float)
df_ani_corr = pd.read_csv('../../data/processed/fastani_output/df_ani_corr.csv', dtype='object').set_index('genome1').astype(float)

df_ani_corr_dist.index = df_ani_corr_dist.index.astype(str)
df_ani_corr_dist.columns = df_ani_corr_dist.columns.astype(str)

df_ani_square.index = df_ani_square.index.astype(str)
df_ani_square.columns = df_ani_square.columns.astype(str)

## Filter by scrubbed genomes

Based on any cleaning that may have been done in `2a`

In [ ]:
scrubbed_strains = scrubbed_metadata.genome_id.astype('str')

good_genomes = df_ani_square[df_ani_square.isna().sum() < .1 * df_ani_square.shape[0]].index # remove genomes which have more than 10% nan values (too distant for fastani)
scrubbed_new = []
for x in scrubbed_strains:
    if x in good_genomes:
        scrubbed_new.append(x)
scrubbed_strains = scrubbed_new    

df_ani_square = df_ani_square.loc[scrubbed_strains, scrubbed_strains]
df_ani_corr = df_ani_corr.loc[scrubbed_strains, scrubbed_strains]
df_ani_corr_dist = df_ani_corr_dist.loc[scrubbed_strains, scrubbed_strains]

## Filter strains by ani distance

- __Criteria 1:__ ani value of 0.95 (soft-limit on bacterial species delineation) not needed for distinction in the genus
- __Criteria 2:__ Any clear outliers


In [ ]:
sns.histplot(df_ani_square.values.flatten())

### Find your Reference/Representative Strain ID (for filtration)


#### note from Josh: need to determine what the representative strain for the genus is, as well as looking at the above plot in order to determine what is occuring there and if any changes to the data need to be made, reference strains below pulled from BV-BRC list and excludes those references not found in the ani matrix

In [ ]:
repr_strains = ["550.3788","158836.1174", "1812935.7", "2478464.3", "299767.18", "539813.36", "2494702.15", "69218.53", "881260.71", "1400147.3", '550.2510', '2494701.30']

In [ ]:
# This cutoff is dependent on the data you see above
# Past studies have gone down as low as 98.5th percentile
# but 99th or 99.9th percentiles are also acceptable
cutoffs = []

for strain in repr_strains:
    cutoffs.append(np.quantile(df_ani_square.loc[strain], 0.01))

cutoff = sum(cutoffs)/len(cutoffs)

# # alternative cutoff using max of possible values
# cutoff = max(cutoffs)


cutoff

In [ ]:
for repr_strain in repr_strains:
    cond = df_ani_square.loc[repr_strain] > cutoff
    good_strains = df_ani_square.loc[repr_strain][cond].index
    
    df_ani_square = df_ani_square.loc[good_strains, good_strains]
    df_ani_corr = df_ani_corr.loc[good_strains, good_strains]
    df_ani_corr_dist = df_ani_corr_dist.loc[good_strains, good_strains]
    
df_ani_corr_dist.shape

In [ ]:
ani_scrubbed_summary = scrubbed_metadata.set_index('genome_id').loc[sorted(df_ani_square.index)].reset_index()
ani_scrubbed_metadata = scrubbed_metadata.set_index('genome_id').loc[sorted(df_ani_square.index)].reset_index()


display(
    ani_scrubbed_metadata.shape,
    ani_scrubbed_metadata.head()
)

## Useful functions for later analysis

In [ ]:
def cluster_corr_dist(df_ani_corr_dist, thresh=0.1, method='ward', metric='euclidean'):
    '''
    Hierarchically ani-based pairwise-pearson-distance matrix
    '''
    link = hc.linkage(sp.distance.squareform(df_ani_corr_dist), method=method, metric=metric)
    dist = sp.distance.squareform(df_ani_corr_dist)
    
    clst = pd.DataFrame(index=df_ani_corr_dist.index)
    clst['cluster'] = hc.fcluster(link, thresh * dist.max(), 'distance')
    
    return link, dist, clst


def remove_bad_strains(df_ani_scd, bad_strains_list):
    good_strains_list = sorted(set(df_ani_scd.index) - set(bad_strains_list))
    
    return df_ani_scd.loc[good_strains_list, good_strains_list]


# Sensitivity analysis to pick the threshold (for E. coli we use 0.1)
# We pick the threshold where the curve just starts to bottom out
def sensitivity_analysis(df_ani_corr_dist_complete):
    x = list(np.logspace(-3, -1, 10)) + list(np.linspace(0.1, 1, 19))
    
    def num_uniq_clusters(thresh):
        link = hc.linkage(sp.distance.squareform(df_ani_corr_dist_complete), method='ward', metric='euclidean')
        dist = sp.distance.squareform(df_ani_corr_dist_complete)
        
        clst = pd.DataFrame(index=df_ani_corr_dist_complete.index)
        clst['cluster'] = hc.fcluster(link, thresh * dist.max(), 'distance')
        
        return len(clst.cluster.unique())
    
    tmp = pd.DataFrame()
    tmp['threshold'] = pd.Series(x)
    tmp['num_clusters'] = pd.Series(x).apply(num_uniq_clusters)
    
    # Find which value the elbow corresponds to
    df_temp = tmp.sort_values(by='num_clusters', ascending=True).reset_index(drop=True)
    
    # transform input into form necessary for package
    results_itr = zip(list(df_temp.index), list(df_temp.num_clusters))
    data = list(results_itr)
    
    rotor = Rotor()
    rotor.fit_rotate(data)
    elbow_idx = rotor.get_elbow_index()
    df_temp['num_clusters'][elbow_idx]
    contamination_cutoff = df_temp['num_clusters'][elbow_idx]
    
    # Grab elbow threshold
    cond = tmp['num_clusters'] == df_temp['num_clusters'][elbow_idx]
    elbow_threshold = tmp[cond]['threshold'].iloc[0]
    
    return tmp, df_temp, elbow_idx, elbow_threshold



## Find threshold for ani clustering

In [ ]:
# Only looking at Complete sequences
cond = scrubbed_summary.genome_status == 'Complete'
complete_seqs = set(scrubbed_summary[cond].genome_id)
complete_seqs = sorted(
    complete_seqs.intersection(set(df_ani_square.index))
)


df_ani_square_complete = df_ani_square.loc[complete_seqs, complete_seqs]
df_ani_corr_complete = df_ani_corr.loc[complete_seqs, complete_seqs]
df_ani_corr_dist_complete = df_ani_square.loc[complete_seqs, complete_seqs]

df_ani_corr_dist_complete.shape

In [ ]:
np.fill_diagonal(df_ani_dist_complete.values,0)

In [ ]:
# Initial sensitivity analysis (gives min val to consider)
df_ani_dist_complete = 1 - df_ani_square_complete

for i in range(min(len(df_ani_dist_complete), len(df_ani_dist_complete.columns))):
    df_ani_dist_complete.iloc[i, i] = 0

tmp, df_temp, elbow_idx, elbow_threshold = sensitivity_analysis(df_ani_dist_complete)

# Plot (tells us to pick something > 0.25)
plt.rcParams["figure.dpi"] = 200
fig, axs = plt.subplots(figsize=(4,3),)
axs.plot(tmp['threshold'], tmp['num_clusters'])
plt.axhline(y=df_temp['num_clusters'][elbow_idx], c="#ff00ff", linestyle='--')
axs.set_ylabel('num_clusters')
axs.set_xlabel('index')
fig.suptitle(
    f"Num clusters decelerates \nafter a value of {df_temp['num_clusters'][elbow_idx]} (threshold: {elbow_threshold})",
    y=1
)
plt.show()

In [ ]:
px.line(tmp, x='threshold', y='num_clusters')

## Plot initial clustermap of ani values

In [ ]:
clst

In [ ]:
elbow_threshold = elbow_threshold+.1 # "round" up

link, dist, clst = cluster_corr_dist(df_ani_dist_complete, thresh=elbow_threshold)

# Color each cluster
cm = matplotlib.colormaps.get_cmap('tab20')
clr = dict(zip(sorted(clst.cluster.unique()), cm.colors+cm.colors+cm.colors))
clst['color'] = clst.cluster.map(clr)

print('Number of colors: ', len(clr))
print('Number of clusters', len(clst.cluster.unique()))

In [ ]:
size = 6

legend_TN = [patches.Patch(color=c, label=l) for l,c in clr.items()]

sns.set(rc={'figure.facecolor':'white'})
g = sns.clustermap(
    df_ani_square_complete,
    figsize=(size,size),
    row_linkage=link,
    col_linkage=link,
    col_colors=clst.color,
    yticklabels=False,
    xticklabels=False,
    cmap='BrBG',
    robust=True,
    center = .95
)

l2=g.ax_heatmap.legend(loc='upper left', bbox_to_anchor=(1.01,0.85), handles=legend_TN,frameon=True)
l2.set_title(title='Clusters',prop={'size':10})

# Compare Mash to FastANI

In [ ]:
df_ani_square = pd.read_csv('../../data/processed/fastani_output/df_ani_square.csv', dtype='object').set_index('genome1').astype(float)
df_mash_square = pd.read_csv('../../data/processed/df_mash_square.csv', dtype='object').set_index('genome1').astype(float).loc[df_ani_square.index,df_ani_square.index]

In [ ]:
scrubbed_strains = scrubbed_metadata.genome_id.astype('str')

good_genomes = df_ani_square[df_ani_square.isna().sum() < .1 * df_ani_square.shape[0]].index # remove genomes which have more than 10% nan values (too distant for fastani)
scrubbed_new = []
for x in scrubbed_strains:
    if x in good_genomes:
        scrubbed_new.append(x)
scrubbed_strains = scrubbed_new    

In [ ]:
df_ani_square = df_ani_square.loc[scrubbed_strains, scrubbed_strains]
df_mash_square = df_mash_square.loc[scrubbed_strains, scrubbed_strains]

In [ ]:
df_ani_square_dist = 1 - df_ani_square # convert to dissimilarity matrix for comparison to mash

In [ ]:
ani_values = df_ani_square_dist.values
mash_values = df_mash_square.values

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
n = len(ani_values)

inds1 = list(range(n))
inds2 = list(range(n))
    
X = ani_values[inds1]
Y = mash_values[inds2]

X_centered = X - X.mean(axis=1, keepdims=True)
Y_centered = Y - Y.mean(axis=1, keepdims=True)
corr_num = np.sum(X_centered * Y_centered, axis=1)
corr_den = np.sqrt(np.sum(X_centered**2, axis=1) * np.sum(Y_centered**2, axis=1))
corrs = corr_num / corr_den

cos_sims = np.sum(X * Y, axis=1) / (np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1))

In [ ]:
corr_mean_true = np.nanmean(corrs)
similarity_mean_true = np.nanmean(cos_sims)

In [ ]:
print(f"Mean correlation of all vectors: {np.nanmean(corrs)}")
print(f"Mean similarity of all vectors: {np.nanmean(cos_sims)}")

In [ ]:
correlations_permuted_saved = {}
similarities_permuted_saved = {}

n = len(ani_values)

for i in tqdm(range(1000)):
    inds1 = np.random.permutation(n)
    inds2 = np.random.permutation(n)
    
    X = ani_values[inds1]
    Y = mash_values[inds2]
    
    X_centered = X - X.mean(axis=1, keepdims=True)
    Y_centered = Y - Y.mean(axis=1, keepdims=True)
    corr_num = np.sum(X_centered * Y_centered, axis=1)
    corr_den = np.sqrt(np.sum(X_centered**2, axis=1) * np.sum(Y_centered**2, axis=1))
    corrs = corr_num / corr_den
    
    cos_sims = np.sum(X * Y, axis=1) / (np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1))
    
    correlations_permuted_saved[i] = np.nanmean(corrs)
    similarities_permuted_saved[i] = np.nanmean(cos_sims)

In [ ]:
p_val_corr = np.sum(list(correlations_permuted_saved.values()) > corr_mean_true) / len(correlations_permuted_saved)
p_val_corr

In [ ]:
p_val_sim = np.sum(list(similarities_permuted_saved.values()) > similarity_mean_true) / len(similarities_permuted_saved)
p_val_sim

In [ ]:
plt.figure(figsize=(6, 3));
plt.hist(list(similarities_permuted_saved.values()));
plt.axvline(similarity_mean_true, c='red', linestyle = '--')
plt.text(similarity_mean_true - .001 , 200, 'True Cosine\nSimilarity\nBetween Strains', fontsize = 10, horizontalalignment='right');
plt.title("Permuted Strains Cosine Similarity vs True Cosine Similarity\nBetween Mash and FastANI Results");
plt.xlabel('Cosine Similarity');
plt.ylabel('Count');